In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
 
from scipy.stats import norm
from scipy.optimize import minimize

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [8]:
# verified variables
vars_verif = [
    'Fairbanks_T2_1d_max',
    'Fairbanks_T2_1h_max',
    'Fairbanks_T2_30d_max',
    'Fairbanks_T2_7d_max',
    'Fort_Bragg_PRECT_1d_max',
    'Fort_Bragg_PRECT_3d_max',
    'Fort_Bragg_PRECT_5d_max',
    'Fort_Bragg_SPEI_09_mean',
    'Fort_Bragg_SPEI_09_min',
    'Fort_Bragg_SPEI_48_mean',
    'Fort_Bragg_SPEI_48_min',
    'Guam_SPEI_03_mean',
    'Guam_SPEI_03_min',
    'Guam_SPEI_48_mean',
    'Guam_SPEI_48_min',
    'Pituffik_FT_days',
    'Pituffik_MDD',
    'Yuma_PG_PRECT_1d_max',
    'Yuma_PG_PRECT_7d_max',
    'Yuma_PG_SPEI_24_mean',
    'Yuma_PG_SPEI_24_min',
    'Yuma_PG_SPEI_48_mean',
    'Yuma_PG_SPEI_48_min',
]

years_verif = np.arange(2000, 2026)
years_train = np.arange(1959, 2000)

data_dir = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/'
save_dir = '/glade/derecho/scratch/ksha/EPRI_data/PP_calib/BMA/'

ds_target = xr.open_zarr(os.path.join(data_dir, 'STN_ERA5_ALL.zarr'))[vars_verif]

# ds_target_hist for determine percentiles
ds_target_hist = ds_target.sel(valid_year=years_train)

# ds_target with dims of `(valid_year: 26)`, valid_year = init_year + lead_year, up to 2025
ds_target = ds_target.sel(valid_year=years_verif)[vars_verif]

ds_dress = xr.open_zarr(save_dir+'Dress.zarr')
ds_dress = ds_dress.rename({'dressed_member': 'member'})
ds_dress_qm = xr.open_zarr(save_dir+'Dress_QM.zarr')
ds_dress_qm = ds_dress_qm.rename({'dressed_member': 'member'})

ds_calib = xr.open_zarr(save_dir+'Calib.zarr')

ds_dress_qm_mean = ds_dress_qm.mean('member')
ds_dress_qm_mean = ds_dress_qm_mean.rename({v: f"{v}_mean" for v in ds_dress_qm_mean.data_vars})

ds_dress_qm_std = ds_dress_qm.std('member')
ds_dress_qm_std = ds_dress_qm_std.rename({v: f"{v}_std" for v in ds_dress_qm_std.data_vars})

ds_calib_qm = xr.merge([ds_dress_qm_mean, ds_dress_qm_std])

# ds_dress_all with dims of `(init_year: 21, lead_year: 10, member: 50)` and 23 post-processed variables
ds_dress_all = xr.merge([ds_dress, ds_dress_qm])[vars_verif]

# ds_calib_all with dims of `(init_year: 21, lead_year: 10)` and 46 variables, which are the `_mu` and `_sigma` of each post-processed variables.
ds_calib_all = xr.merge([ds_calib, ds_calib_qm])

### Verification Scores

In [10]:
# ============================================================================
# 4. ACC and CRPS per (variable, lead_year)
# ============================================================================
def gather_pairs_at_lead(da_fc, da_obs, il, has_member=True):
    """Forecast/obs pairs for a single lead_year index `il`.
    Returns fc shape (N, M) or (N,), obs shape (N,)."""
    init_years = da_fc['init_year'].values
    lead       = int(da_fc['lead_year'].values[il])
    valid_yrs  = init_years + lead

    obs_avail = da_obs['valid_year'].values
    in_mask   = np.isin(valid_yrs, obs_avail)
    if not in_mask.any():
        return (np.empty((0, da_fc.sizes['member'])) if has_member
                else np.empty(0)), np.empty(0)

    obs_arr = da_obs.sel(valid_year=valid_yrs[in_mask]).values

    if has_member:
        fc_arr = da_fc.isel(lead_year=il).values[in_mask, :]
        finite = np.all(np.isfinite(fc_arr), axis=1) & np.isfinite(obs_arr)
        return fc_arr[finite, :], obs_arr[finite]
    else:
        fc_arr = da_fc.isel(lead_year=il).values[in_mask]
        finite = np.isfinite(fc_arr) & np.isfinite(obs_arr)
        return fc_arr[finite], obs_arr[finite]


def crps_ensemble_pooled(ens, obs):
    if ens.size == 0:
        return np.nan
    term1 = np.mean(np.abs(ens - obs[:, None]), axis=1)
    diff  = np.abs(ens[:, :, None] - ens[:, None, :])
    term2 = 0.5 * diff.mean(axis=(1, 2))
    return float(np.mean(term1 - term2))


def acc_pooled(fc_mean, obs, clim):
    if fc_mean.size < 2 or not np.isfinite(clim):
        return np.nan
    fa = fc_mean - clim
    oa = obs     - clim
    den = np.sqrt(np.sum(fa * fa) * np.sum(oa * oa))
    return float(np.sum(fa * oa) / den) if den > 0 else np.nan


lead_years = ds_dress_all['lead_year'].values
n_lead = len(lead_years)
n_var  = len(vars_verif)

crps_arr    = np.full((n_var, n_lead), np.nan)
acc_calib   = np.full((n_var, n_lead), np.nan)
acc_dressmn = np.full((n_var, n_lead), np.nan)
n_eff_arr   = np.zeros((n_var, n_lead), dtype=np.int64)
clim_arr    = np.full(n_var, np.nan)
bias_arr = np.full((n_var, n_lead), np.nan)

for iv, var in enumerate(tqdm(vars_verif, desc='ACC / CRPS     ')):
    # Climatology (single value per variable, from 1959-1999 ERA5)
    hist = ds_target_hist[var].values
    hist = hist[np.isfinite(hist)]
    clim = float(hist.mean()) if hist.size else np.nan
    clim_arr[iv] = clim

    for il in range(n_lead):
        # CRPS from dressed ensemble
        fc_ens, obs = gather_pairs_at_lead(ds_dress_all[var],
                                           ds_target[var], il, has_member=True)
        crps_arr[iv, il]  = crps_ensemble_pooled(fc_ens, obs)
        n_eff_arr[iv, il] = int(obs.size)

        # ACC from ds_calib_all _mu
        mu_flat, obs_mu = gather_pairs_at_lead(ds_calib_all[f'{var}_mean'],
                                               ds_target[var], il, has_member=False)
        acc_calib[iv, il] = acc_pooled(mu_flat, obs_mu, clim)

        # ACC from dressed ensemble mean (sanity check)
        if fc_ens.size:
            acc_dressmn[iv, il] = acc_pooled(fc_ens.mean(axis=1), obs, clim)
            bias_arr[iv, il] = float(fc_ens.mean(axis=1).mean() - obs.mean())


ds_verif = xr.Dataset(
    {
        'CRPS':           (('variable', 'lead_year'), crps_arr),
        'ACC_calib':      (('variable', 'lead_year'), acc_calib),
        'ACC_dress_mean': (('variable', 'lead_year'), acc_dressmn),
        'n_pairs':        (('variable', 'lead_year'), n_eff_arr),
        'climatology':    (('variable',),             clim_arr),
        'Bias':           (('variable', 'lead_year'), bias_arr),
    },
    coords={'variable': vars_verif, 'lead_year': lead_years},
)
ds_verif['CRPS'].attrs['description']           = 'Empirical CRPS of 50-member dressed ensemble vs ERA5, per lead_year'
ds_verif['ACC_calib'].attrs['description']      = 'ACC of calibrated mean vs ERA5, climatology from 1959-1999, per lead_year'
ds_verif['ACC_dress_mean'].attrs['description'] = 'ACC of dressed ensemble mean vs ERA5, per lead_year'
ds_verif['climatology'].attrs['description']    = 'ERA5 1959-1999 mean used as ACC reference'
ds_verif['Bias'].attrs['description'] = 'Mean bias: mean(ensemble_mean) - mean(obs), per lead_year'

ds_verif.to_zarr(save_dir+'/verif_scores.zarr', mode='w')
print(save_dir+'/verif_scores.zarr')

ACC / CRPS     : 100%|██████████| 23/23 [00:04<00:00,  5.61it/s]


/glade/derecho/scratch/ksha/EPRI_data/PP_calib/BMA//verif_scores.zarr


In [11]:
# Quick summary
print("\nPer-lead CRPS:")
print(ds_verif['CRPS'].to_pandas().round(3))
print("\nPer-lead ACC_calib:")
print(ds_verif['ACC_calib'].to_pandas().round(3))


Per-lead CRPS:
lead_year                     0       1       2       3       4       5  \
variable                                                                  
Fairbanks_T2_1d_max       0.965   0.881   0.987   1.050   1.241   0.913   
Fairbanks_T2_1h_max       1.129   1.051   1.219   1.271   1.133   1.055   
Fairbanks_T2_30d_max      0.834   0.835   1.018   0.916   0.794   0.790   
Fairbanks_T2_7d_max       0.803   0.809   0.927   0.863   0.808   0.740   
Fort_Bragg_PRECT_1d_max  15.734  16.651  17.943  17.591  22.755  20.135   
Fort_Bragg_PRECT_3d_max   9.414   9.879   9.705   9.814  11.230  10.574   
Fort_Bragg_PRECT_5d_max   6.431   6.429   6.690   6.250   7.178   6.685   
Fort_Bragg_SPEI_09_mean   0.499   0.565   0.572   0.541   0.517   0.567   
Fort_Bragg_SPEI_09_min    0.606   0.611   0.626   0.590   0.564   0.608   
Fort_Bragg_SPEI_48_mean   0.841   1.061   1.054   0.946   1.048   1.066   
Fort_Bragg_SPEI_48_min    0.919   1.145   1.100   1.032   1.109   1.287   
Guam_SPEI